# Subtask 2 — Code Repair with Pointer-Generator Transformer

Modules:
- **dataset.py** — `TextSource` with proper START / END / PAD tokens
- **models.py** — Encoder-Decoder Transformer + Pointer-Generator gate
- **baselines.py** — RNN + Self-Attention baseline
- **training.py** — Loss, train/val steps, early stopping
- **inference.py** — Greedy autoregressive decoding & evaluation

## Setup

In [1]:
# BPE tokenizer build command (run once in terminal if .so is missing):
# g++ -O3 -Wall -shared -std=c++14 -fPIC $(python3 -m pybind11 --includes) \
#   "Custom_Tokenizers/bpe_tokenizer.cpp" \
#   -o bpe_tokenizer$(python3-config --extension-suffix)


In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from flax import nnx
import optax
import bpe_tokenizer
import time

from dataset import TextSource, make_loader
from models import TransformerModel, PointerGeneratorTransformer
from baselines import RNNSelfAttentionModel
from training import (
    compute_loss, train_step, validation_step,
    train_step_pointer_gen, validation_step_pointer_gen,
    train_step_baseline, validation_step_baseline,
    EarlyStopping,
)
from inference import greedy_decode, evaluate_autoregressive, evaluate_baseline

RANDOM_SEED = 14

## Data & Tokenizer

In [3]:
# Load datasets
df = pd.read_parquet("./Code_Dataset/train.parquet")
val_df = pd.read_parquet("./Code_Dataset/validation.parquet")
test_df = pd.read_parquet("./Code_Dataset/test.parquet")

# Train BPE tokenizer on the combined corpus
corpus = df['buggy'].str.cat(sep='\u0000') + df['fixed'].str.cat(sep='\u0000')
Tokenizer = bpe_tokenizer.BPETokenizer()
Tokenizer.train(corpus, num_merges=1000)

# Special tokens
VOCAB_SIZE = Tokenizer.get_vocab_size()
PAD_TOKEN = VOCAB_SIZE
START_TOKEN = VOCAB_SIZE + 1
END_TOKEN = VOCAB_SIZE + 2
TOTAL_VOCAB_SIZE = VOCAB_SIZE + 3

# Sequence lengths
max_buggy_len = max(len(Tokenizer.encode(x)) for x in df['buggy'])
max_fixed_len = max(len(Tokenizer.encode(x)) for x in df['fixed'])
MAX_SEQ_LEN = max(max_buggy_len, max_fixed_len)   # content length
FULL_SEQ_LEN = MAX_SEQ_LEN + 2                     # with START + END

print(f"Vocab size: {VOCAB_SIZE}")
print(f"Total vocab size (with special tokens): {TOTAL_VOCAB_SIZE}")
print(f"Max content length: {MAX_SEQ_LEN}")
print(f"Full sequence length: {FULL_SEQ_LEN}")

Vocab size: 1256
Total vocab size (with special tokens): 1259
Max content length: 261
Full sequence length: 263


In [4]:
# Create DataLoaders
BATCH_SIZE = 128

train_loader = make_loader(
    df['buggy'], df['fixed'], Tokenizer, MAX_SEQ_LEN,
    PAD_TOKEN, START_TOKEN, END_TOKEN,
    batch_size=BATCH_SIZE, training=True, seed=RANDOM_SEED,
)
val_loader = make_loader(
    val_df['buggy'], val_df['fixed'], Tokenizer, MAX_SEQ_LEN,
    PAD_TOKEN, START_TOKEN, END_TOKEN,
    batch_size=BATCH_SIZE, training=False, seed=RANDOM_SEED,
)
test_loader = make_loader(
    test_df['buggy'], test_df['fixed'], Tokenizer, MAX_SEQ_LEN,
    PAD_TOKEN, START_TOKEN, END_TOKEN,
    batch_size=BATCH_SIZE, training=False, seed=RANDOM_SEED,
)

print("Dataloaders successfully created.")

Dataloaders successfully created.


## Sanity Check — Dataset

In [5]:
# Verify first sample from train loader
sample = next(iter(train_loader))
x_sample, y_sample = sample['input'], sample['output']
print(f"Input batch shape: {x_sample.shape}")   # (BATCH, FULL_SEQ_LEN)
print(f"Output batch shape: {y_sample.shape}")

row = 0
assert int(x_sample[row, 0]) == START_TOKEN, "First token must be START"
assert int(y_sample[row, 0]) == START_TOKEN, "First token must be START"
print("\u2714 START token in position 0")

# Find END token position in target
end_pos = int(jnp.argmax(y_sample[row] == END_TOKEN))
assert end_pos > 0, "END token must be present"
print(f"\u2714 END token found at position {end_pos}")

# Everything after END should be PAD
assert jnp.all(y_sample[row, end_pos+1:] == PAD_TOKEN), "Post-END must be PAD"
print("\u2714 Padding is correct after END")

Input batch shape: (128, 263)
Output batch shape: (128, 263)
✔ START token in position 0
✔ END token found at position 105
✔ Padding is correct after END


## Model Setup

Instantiate the **Pointer-Generator Transformer** and its optimiser.

In [6]:
rngs = nnx.Rngs(params=0, dropout=1)

# Hyperparameters
d_model = 128
n_heads = 4
num_layers = 2

model = PointerGeneratorTransformer(
    vocab_size=TOTAL_VOCAB_SIZE,
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    max_seq_len=FULL_SEQ_LEN,
    pad_token=PAD_TOKEN,
    start_token=START_TOKEN,
    rngs=rngs,
)

# Learning-rate schedule + optimiser
lr = 1e-3
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=lr,
    warmup_steps=100,
    decay_steps=2000,
    end_value=1e-5,
)
optimizer = nnx.Optimizer(
    model,
    optax.adamw(schedule, weight_decay=1e-4),
    wrt=nnx.Param,
)

print(f"Pointer-Generator Transformer initialised  |  d={d_model}  h={n_heads}  L={num_layers}")

Pointer-Generator Transformer initialised  |  d=128  h=4  L=2


## Training — Pointer-Generator Transformer

> **GPU-intensive cell.**  Runs the full training loop.

In [7]:
es = EarlyStopping(patience=5)
epochs = 15

print("Starting training of Pointer-Generator Transformer...")
for epoch in range(epochs):
    t0 = time.time()
    train_losses = []

    for batch in train_loader:
        total_loss = train_step_pointer_gen(
            model, optimizer, batch, PAD_TOKEN,
        )
        train_losses.append(float(total_loss))

    val_losses = []
    for batch in val_loader:
        val_loss = validation_step_pointer_gen(model, batch, PAD_TOKEN)
        val_losses.append(float(val_loss))

    mean_train = np.mean(train_losses)
    mean_val = np.mean(val_losses)
    elapsed = time.time() - t0

    print(f"Epoch {epoch+1:02d} | Train: {mean_train:.4f} | Val: {mean_val:.4f} | Time: {elapsed:.2f}s")

    if es.step(mean_val, model):
        print("Early stopping. Restoring best weights.")
        break

if es.best_state is not None:
    nnx.update(model, es.best_state)
    print("Restored best model parameters.")

Starting training of Pointer-Generator Transformer...
Epoch 01 | Train: 0.9472 | Val: 0.3413 | Time: 164.17s
Epoch 02 | Train: 0.2517 | Val: 0.1967 | Time: 50.76s
Epoch 03 | Train: 0.1757 | Val: 0.1619 | Time: 50.70s
Epoch 04 | Train: 0.1529 | Val: 0.1500 | Time: 50.74s
Epoch 05 | Train: 0.1440 | Val: 0.1472 | Time: 50.62s
Epoch 06 | Train: 0.1423 | Val: 0.1468 | Time: 51.22s
Epoch 07 | Train: 0.1418 | Val: 0.1461 | Time: 50.56s
Epoch 08 | Train: 0.1413 | Val: 0.1458 | Time: 50.75s
Epoch 09 | Train: 0.1409 | Val: 0.1454 | Time: 51.52s


Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=2):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=2):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=2):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=2):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=2):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=2):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/aneesh_shastri/miniforge3/envs/ml_env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/aneesh_shastri/miniforge3/envs/ml_env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/aneesh_shastri/miniforge3/envs/ml_env/

KeyboardInterrupt: 

## Evaluation — Autoregressive Greedy Decoding

No teacher forcing: the model generates each token conditioned only
on its own previous predictions.

In [8]:
print("Evaluating with autoregressive (greedy) decoding...")
max_decode_len = FULL_SEQ_LEN - 1

test_token_acc, test_seq_acc = evaluate_autoregressive(
    model, test_loader,
    START_TOKEN, END_TOKEN, PAD_TOKEN,
    max_decode_len,
)
print(f"Test Token Accuracy:           {test_token_acc:.4f}")
print(f"Test Exact Sequence Accuracy:   {test_seq_acc:.4f}")

Evaluating with autoregressive (greedy) decoding...
Test Token Accuracy:           0.5133
Test Exact Sequence Accuracy:   0.0041


## Qualitative Examples

In [ ]:
# Grab a small batch and decode
sample_batch = next(iter(test_loader))
X_show = sample_batch['input'][:4]
y_show = sample_batch['output'][:4]

generated = greedy_decode(
    model, X_show, START_TOKEN, END_TOKEN, PAD_TOKEN, max_decode_len,
)

def _extract(seq, special={PAD_TOKEN, START_TOKEN, END_TOKEN}):
    """Extract content tokens up to END (exclusive), skipping special."""
    out = []
    for t in seq:
        t = int(t)
        if t == END_TOKEN or t == PAD_TOKEN:
            break
        if t not in special:
            out.append(t)
    return out

for i in range(min(4, len(X_show))):
    inp_tok = _extract(X_show[i])
    tgt_tok = _extract(y_show[i])
    gen_tok = _extract(generated[i])

    print(f"\n--- Example {i+1} ---")
    print(f"Input (buggy):  {Tokenizer.decode(inp_tok)[:120]}...")
    print(f"Target (fixed): {Tokenizer.decode(tgt_tok)[:120]}...")
    print(f"Generated:      {Tokenizer.decode(gen_tok)[:120]}...")


--- Example 1 ---
Input (buggy):  public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) ) ) { return VAR_2 ....
Target (fixed): public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) ) ) { return VAR_1 ....
Generated:      public void METHOD_1 instanceof,.println METHOD_2 METHOD_2 METHODlllllloneneneneeenenes elseeennnoalllll synchronized in...

--- Example 2 ---
Input (buggy):  protected void METHOD_1 ( boolean VAR_1 ) { VAR_2 . METHOD_2 ( data . METHOD_3 ( VAR_3 . getString ( VAR_4 ) ) . getStri...
Target (fixed): protected void METHOD_1 ( boolean VAR_1 ) { VAR_2 . METHOD_2 ( data . METHOD_3 ( VAR_3 . getString ( VAR_4 ) ) . getStri...
Generated:      public boolean METHOD_2ninal boolean type instanceofinal returninal return if instanceof if instanceof>snalinal java.lan...

--- Example 3 ---
Input (buggy):  public java.util.Map < java.lang.String , java.lang.Object > METHOD_1 ( java.util.Map <

---
## Baseline: RNN + Self-Attention (Optional)

Non-autoregressive baseline that directly maps each input position
to an output token.  Uncomment and run to compare.

In [ ]:
"""
# ---- Baseline model ----
rnn_model = RNNSelfAttentionModel(
    vocab_size=TOTAL_VOCAB_SIZE,
    d_model=d_model,
    hidden_dim=d_model,
    n_heads=n_heads,
    max_seq_len=FULL_SEQ_LEN,
    pad_token=PAD_TOKEN,
    rngs=rngs,
)
rnn_optimizer = nnx.Optimizer(
    rnn_model,
    optax.adamw(schedule, weight_decay=1e-4),
    wrt=nnx.Param,
)
es_rnn = EarlyStopping(patience=5)

print("Starting training of RNN+SelfAttention baseline...")
for epoch in range(epochs):
    t0 = time.time()
    losses = []
    for batch in train_loader:
        loss = train_step_baseline(rnn_model, rnn_optimizer, batch, PAD_TOKEN)
        losses.append(float(loss))
    vloss = []
    for batch in val_loader:
        vloss.append(float(validation_step_baseline(rnn_model, batch, PAD_TOKEN)))
    mt, mv = np.mean(losses), np.mean(vloss)
    print(f"Epoch {epoch+1:02d} | Train: {mt:.4f} | Val: {mv:.4f} | Time: {time.time()-t0:.2f}s")
    if es_rnn.step(mv, rnn_model):
        print("Early stopping.")
        break
if es_rnn.best_state is not None:
    nnx.update(rnn_model, es_rnn.best_state)

tok_acc, seq_acc = evaluate_baseline(rnn_model, test_loader, PAD_TOKEN)
print(f"Baseline Token Acc: {tok_acc:.4f}  |  Seq Acc: {seq_acc:.4f}")
"""


'\n# ---- Baseline model ----\nrnn_model = RNNSelfAttentionModel(\n    vocab_size=TOTAL_VOCAB_SIZE,\n    d_model=d_model,\n    hidden_dim=d_model,\n    n_heads=n_heads,\n    max_seq_len=FULL_SEQ_LEN,\n    pad_token=PAD_TOKEN,\n    rngs=rngs,\n)\nrnn_optimizer = nnx.Optimizer(\n    rnn_model,\n    optax.adamw(schedule, weight_decay=1e-4),\n    wrt=nnx.Param,\n)\nes_rnn = EarlyStopping(patience=5)\n\nprint("Starting training of RNN+SelfAttention baseline...")\nfor epoch in range(epochs):\n    t0 = time.time()\n    losses = []\n    for batch in train_loader:\n        loss = train_step_baseline(rnn_model, rnn_optimizer, batch, PAD_TOKEN)\n        losses.append(float(loss))\n    vloss = []\n    for batch in val_loader:\n        vloss.append(float(validation_step_baseline(rnn_model, batch, PAD_TOKEN)))\n    mt, mv = np.mean(losses), np.mean(vloss)\n    print(f"Epoch {epoch+1:02d} | Train: {mt:.4f} | Val: {mv:.4f} | Time: {time.time()-t0:.2f}s")\n    if es_rnn.step(mv, rnn_model):\n        pr